In [19]:
import common
from db import get_engine
engine = get_engine()

* Xam cedvellerin yuklenmesi

In [2]:
airports_raw=common.pd.read_csv("/workspaces/Flight-delay-analysis-E2E/data/Flight/airports.csv")
airlines_raw=common.pd.read_csv("/workspaces/Flight-delay-analysis-E2E/data/Flight/airlines.csv")
flights_raw=common.pd.read_parquet('/workspaces/Flight-delay-analysis-E2E/data/Flight/flights.parquet')
weather_raw=common.pd.read_parquet('/workspaces/Flight-delay-analysis-E2E/data/api/weather_2015_all.parquet')

In [3]:
print("--- FLIGHTS ---")
print(flights_raw.info())
print(flights_raw.head(2))

print("\n--- AIRPORTS ---")
print(airports_raw.info())
print(airports_raw.head(2))

print("\n--- AIRLINES ---")
print(airlines_raw.info())
print(airlines_raw.head(2))

print("\n--- WEATHER ---")
print(weather_raw.info())
print(weather_raw.head(2))

--- FLIGHTS ---
<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 31 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int16  
 1   MONTH                int8   
 2   DAY                  int8   
 3   DAY_OF_WEEK          int8   
 4   AIRLINE              string 
 5   FLIGHT_NUMBER        int32  
 6   TAIL_NUMBER          string 
 7   ORIGIN_AIRPORT       string 
 8   DESTINATION_AIRPORT  string 
 9   SCHEDULED_DEPARTURE  int16  
 10  DEPARTURE_TIME       float32
 11  DEPARTURE_DELAY      float32
 12  TAXI_OUT             float32
 13  WHEELS_OFF           float32
 14  SCHEDULED_TIME       float32
 15  ELAPSED_TIME         float32
 16  AIR_TIME             float32
 17  DISTANCE             int16  
 18  WHEELS_ON            float32
 19  TAXI_IN              float32
 20  SCHEDULED_ARRIVAL    int16  
 21  ARRIVAL_TIME         float32
 22  ARRIVAL_DELAY        float32
 23  DIVERTED             int8  

* 5 reqemli aeroport kodlarinin 3 herfli koda cevrilmesi

In [4]:


# 1. Rəsmi BTS (DOT Airport ID -> IATA) xəritəsini oxuyuruq
bts_mapping_url = "https://gist.githubusercontent.com/patil-charudatta/e5cb878df0288898114f2e54eb8ff9c9/raw/dot_airport_id_mapping.csv"
df_bts = common.pd.read_csv(bts_mapping_url, dtype=str)

# 2. Xəritələmə lüğətini yaradırıq (Code -> IATA)
dot_to_iata = dict(zip(df_bts['Code'].str.strip(), df_bts['IATA'].str.strip()))

# 3. flights_raw cədvəlindəki sütunları təmizləyib birbaşa əvəzləyirik
flights_raw['ORIGIN_AIRPORT'] = flights_raw['ORIGIN_AIRPORT'].astype(str).str.strip().replace(dot_to_iata)
flights_raw['DESTINATION_AIRPORT'] = flights_raw['DESTINATION_AIRPORT'].astype(str).str.strip().replace(dot_to_iata)

# 4. Yoxlama: Hələ də çevrilməmiş rəqəmli kod qaldımı?
remaining_origin = flights_raw[flights_raw['ORIGIN_AIRPORT'].str.isnumeric()]['ORIGIN_AIRPORT'].unique()
remaining_dest = flights_raw[flights_raw['DESTINATION_AIRPORT'].str.isnumeric()]['DESTINATION_AIRPORT'].unique()

print(f"ORIGIN üzrə qalan rəqəmli kod sayı: {len(remaining_origin)}")
print(f"DESTINATION üzrə qalan rəqəmli kod sayı: {len(remaining_dest)}")

HTTPError: HTTP Error 404: Not Found

In [5]:
test_new = common.pd.read_parquet(
    "/workspaces/Flight-delay-analysis-E2E/data/Flight/flights_selected.parquet"
)

test_new.shape

(5819079, 23)

In [6]:
import pyarrow.parquet as pq

new_file = pq.ParquetFile(
    "/workspaces/Flight-delay-analysis-E2E/data/Flight/flights_new.parquet"
)

print("Sətir sayı:", new_file.metadata.num_rows)
print("Sütun sayı:", len(new_file.schema.names))
print("Sütunlar:", new_file.schema.names)

Sətir sayı: 5812365
Sütun sayı: 28
Sütunlar: ['MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'SCHEDULED_DEPARTURE_DATETIME']


In [7]:
flights_new = common.pd.read_parquet(
    "/workspaces/Flight-delay-analysis-E2E/data/Flight/flights_new.parquet"
)

print(flights_new.shape)
print(flights_new.columns.tolist())

(5812365, 28)
['MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'SCHEDULED_DEPARTURE_DATETIME']


In [8]:
# 1. FLIGHTS + AIRLINES

flights_joined = flights_new.merge(
    airlines_raw[["IATA_CODE", "AIRLINE"]].rename(
        columns={"AIRLINE": "AIRLINE_NAME"}
    ),
    how="left",
    left_on="AIRLINE",
    right_on="IATA_CODE",
    validate="many_to_one"
)

flights_joined = flights_joined.drop(columns=["IATA_CODE"])

print("Əvvəl:", len(flights_new))
print("Airlines join-dən sonra:", len(flights_joined))
print("Airline adı tapılmayan:", flights_joined["AIRLINE_NAME"].isna().sum())

Əvvəl: 5812365
Airlines join-dən sonra: 5812365
Airline adı tapılmayan: 0


In [9]:
# 2. ORIGIN AIRPORT JOIN

origin_airports = airports_raw[
    ["IATA_CODE", "AIRPORT", "CITY", "STATE", "COUNTRY"]
].rename(columns={
    "IATA_CODE": "ORIGIN_AIRPORT_CODE",
    "AIRPORT": "ORIGIN_AIRPORT_NAME",
    "CITY": "ORIGIN_CITY",
    "STATE": "ORIGIN_STATE",
    "COUNTRY": "ORIGIN_COUNTRY"
})

flights_joined = flights_joined.merge(
    origin_airports,
    how="left",
    left_on="ORIGIN_AIRPORT",
    right_on="ORIGIN_AIRPORT_CODE",
    validate="many_to_one"
)

print("Origin join-dən sonra:", len(flights_joined))
print(
    "Origin airport tapılmayan:",
    flights_joined["ORIGIN_AIRPORT_NAME"].isna().sum()
)

Origin join-dən sonra: 5812365
Origin airport tapılmayan: 1011


# Origin Airport JOIN yoxlaması

Bu mərhələdə `flights_new` cədvəlindəki `ORIGIN_AIRPORT` kodlarını `airports.csv` cədvəli ilə birləşdirdik.

Məqsəd:
- hər uçuşun başladığı aeroport üçün aeroport adı, şəhər, ştat və ölkə məlumatını əlavə etmək;
- join-dən sonra sətir sayının dəyişib-dəyişmədiyini yoxlamaq;
- uyğun gəlməyən aeroport kodlarını tapmaq.

Join-dən əvvəl və sonra sətir sayı eyni qaldı:

- Flights sətir sayı: **5,812,365**
- Origin join-dən sonra: **5,812,365**

Bu yaxşı nəticədir, çünki join datanı çoxaltmadı və heç bir uçuş sətri itməyib.

Amma **1,011 sətirdə airport məlumatı tapılmadı**.

Ona görə növbəti addımda həmin uyğun gəlməyən `ORIGIN_AIRPORT` kodlarını ayrıca yoxlayırıq. Məqsəd problemin:
- rəqəmli DOT airport kodlarından,
- `airports.csv` daxilində olmayan kodlardan,
- və ya başqa data keyfiyyəti problemindən

qaynaqlandığını müəyyən etməkdir.


In [10]:
unmatched_origin = (
    flights_joined[
        flights_joined["ORIGIN_AIRPORT_NAME"].isna()
    ]["ORIGIN_AIRPORT"]
    .value_counts()
)

unmatched_origin

ORIGIN_AIRPORT
NYL    168
Name: count, dtype: int64

### Uyğun gəlməyən Origin Airport kodlarının yoxlanması

Origin Airport join-dən sonra 1,011 sətirdə airport məlumatı tapılmadı.

İlk yoxlamada `NYL` kodunun 168 dəfə uyğun gəlmədiyi görüldü.

`value_counts()` boş dəyərləri standart olaraq göstərmədiyi üçün, qalan uyğun gəlməyən sətrləri də görmək məqsədilə `dropna=False` istifadə edirik.

Bu yoxlamanın məqsədi problemin real airport kodundan, yoxsa boş `ORIGIN_AIRPORT` dəyərlərindən qaynaqlandığını müəyyən etməkdir.

In [11]:
flights_joined.loc[
    flights_joined["ORIGIN_AIRPORT_NAME"].isna(),
    "ORIGIN_AIRPORT"
].value_counts(dropna=False)

ORIGIN_AIRPORT
<NA>    843
NYL     168
Name: count, dtype: int64

### Origin Airport JOIN nəticəsi

Origin Airport join-dən sonra ümumi sətir sayı dəyişmədi:

- Join-dən əvvəl: **5,812,365**
- Join-dən sonra: **5,812,365**

Bu, join zamanı uçuş sətrlərinin itmədiyini və datanın çoxalmadığını göstərir.

Airport məlumatı tapılmayan **1,011 sətir** ayrıca yoxlanıldı:

- **843 sətirdə `ORIGIN_AIRPORT` boşdur (`NA`)**
- **168 sətirdə `ORIGIN_AIRPORT = NYL` kodu var**, lakin bu kod `airports.csv` cədvəlində uyğunlaşmadı.

Bu sətrlər silinmədi və sonrakı analiz üçün saxlanıldı.

In [12]:
print("Yekun sətir sayı:", len(flights_joined))

flights_joined[
    [
        "AIRLINE",
        "AIRLINE_NAME",
        "ORIGIN_AIRPORT",
        "ORIGIN_AIRPORT_NAME",
        "ORIGIN_CITY",
        "ORIGIN_STATE",
        "ORIGIN_COUNTRY"
    ]
].head(10)

Yekun sətir sayı: 5812365


,AIRLINE,AIRLINE_NAME,ORIGIN_AIRPORT,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_COUNTRY
0,AS,Alaska Airlines Inc.,ANC,Ted Stevens Anchorage International Airport,Anchorage,AK,USA
1,AA,American Airlines Inc.,LAX,Los Angeles International Airport,Los Angeles,CA,USA
2,US,US Airways Inc.,SFO,San Francisco International Airport,San Francisco,CA,USA
3,AA,American Airlines Inc.,LAX,Los Angeles International Airport,Los Angeles,CA,USA
4,AS,Alaska Airlines Inc.,SEA,Seattle-Tacoma International Airport,Seattle,WA,USA
5,DL,Delta Air Lines Inc.,SFO,San Francisco International Airport,San Francisco,CA,USA
6,NK,Spirit Air Lines,LAS,McCarran International Airport,Las Vegas,NV,USA
7,US,US Airways Inc.,LAX,Los Angeles International Airport,Los Angeles,CA,USA
8,AA,American Airlines Inc.,SFO,San Francisco International Airport,San Francisco,CA,USA
9,DL,Delta Air Lines Inc.,LAS,McCarran International Airport,Las Vegas,NV,USA


### Destination Airport JOIN

Destination Airport məlumatını Pandas ilə birləşdirməyə çalışdıq.

Lakin dataset təxminən 5.8 milyon sətir olduğu və əvvəlki join-lərdən sonra DataFrame böyüdüyü üçün Destination merge zamanı kernel yaddaş çatışmazlığı səbəbindən dayandı.

Buna görə böyük join əməliyyatlarının SQL Server-də aparılmasına qərar verildi.

In [13]:
# 3. DESTINATION AIRPORT JOIN

destination_airports = airports_raw[
    ["IATA_CODE", "AIRPORT", "CITY", "STATE", "COUNTRY"]
].rename(columns={
    "IATA_CODE": "DESTINATION_AIRPORT_CODE",
    "AIRPORT": "DESTINATION_AIRPORT_NAME",
    "CITY": "DESTINATION_CITY",
    "STATE": "DESTINATION_STATE",
    "COUNTRY": "DESTINATION_COUNTRY"
})

flights_joined = flights_joined.merge(
    destination_airports,
    how="left",
    left_on="DESTINATION_AIRPORT",
    right_on="DESTINATION_AIRPORT_CODE",
    validate="many_to_one"
)

print("Destination join-dən sonra:", len(flights_joined))
print(
    "Destination airport tapılmayan:",
    flights_joined["DESTINATION_AIRPORT_NAME"].isna().sum()
)

: 

In [2]:
import pyarrow.parquet as pq

file_path = "/workspaces/Flight-delay-analysis-E2E/data/Flight/flights_new.parquet"

parquet_file = pq.ParquetFile(file_path)

for i, batch in enumerate(
    parquet_file.iter_batches(batch_size=100_000)
):
    df_batch = batch.to_pandas()

    df_batch.to_sql(
        "flights_new",
        engine,
        schema="bronze",
        if_exists="replace" if i == 0 else "append",
        index=False
    )

    print(f"batch {i} yükləndi")

batch 0 yükləndi
batch 1 yükləndi
batch 2 yükləndi
batch 3 yükləndi
batch 4 yükləndi
batch 5 yükləndi
batch 6 yükləndi
batch 7 yükləndi
batch 8 yükləndi
batch 9 yükləndi
batch 10 yükləndi
batch 11 yükləndi
batch 12 yükləndi
batch 13 yükləndi
batch 14 yükləndi
batch 15 yükləndi
batch 16 yükləndi
batch 17 yükləndi
batch 18 yükləndi
batch 19 yükləndi
batch 20 yükləndi
batch 21 yükləndi
batch 22 yükləndi
batch 23 yükləndi
batch 24 yükləndi
batch 25 yükləndi
batch 26 yükləndi
batch 27 yükləndi
batch 28 yükləndi
batch 29 yükləndi
batch 30 yükləndi
batch 31 yükləndi
batch 32 yükləndi
batch 33 yükləndi
batch 34 yükləndi
batch 35 yükləndi
batch 36 yükləndi
batch 37 yükləndi
batch 38 yükləndi
batch 39 yükləndi
batch 40 yükləndi
batch 41 yükləndi
batch 42 yükləndi
batch 43 yükləndi
batch 44 yükləndi
batch 45 yükləndi
batch 46 yükləndi
batch 47 yükləndi
batch 48 yükləndi
batch 49 yükləndi
batch 50 yükləndi
batch 51 yükləndi
batch 52 yükləndi
batch 53 yükləndi
batch 54 yükləndi
batch 55 yükləndi
ba

In [3]:
count = common.pd.read_sql(
    "SELECT COUNT_BIG(*) AS row_count FROM bronze.flights_new",
    engine
)

count

,row_count
0,5812365


In [4]:
join_check = common.pd.read_sql("""
SELECT COUNT_BIG(*) AS row_count
FROM bronze.flights_new f

LEFT JOIN bronze.airlines a
    ON f.AIRLINE = a.IATA_CODE

LEFT JOIN bronze.airports o
    ON f.ORIGIN_AIRPORT = o.IATA_CODE

LEFT JOIN bronze.airports d
    ON f.DESTINATION_AIRPORT = d.IATA_CODE
""", engine)

join_check

,row_count
0,5812365


In [5]:
join_preview = common.pd.read_sql("""
SELECT TOP 10
    f.AIRLINE,
    a.AIRLINE AS AIRLINE_NAME,

    f.ORIGIN_AIRPORT,
    o.AIRPORT AS ORIGIN_AIRPORT_NAME,
    o.CITY AS ORIGIN_CITY,

    f.DESTINATION_AIRPORT,
    d.AIRPORT AS DESTINATION_AIRPORT_NAME,
    d.CITY AS DESTINATION_CITY

FROM bronze.flights_new f

LEFT JOIN bronze.airlines a
    ON f.AIRLINE = a.IATA_CODE

LEFT JOIN bronze.airports o
    ON f.ORIGIN_AIRPORT = o.IATA_CODE

LEFT JOIN bronze.airports d
    ON f.DESTINATION_AIRPORT = d.IATA_CODE
""", engine)

join_preview

,AIRLINE,AIRLINE_NAME,ORIGIN_AIRPORT,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,DESTINATION_AIRPORT,DESTINATION_AIRPORT_NAME,DESTINATION_CITY
0,AS,Alaska Airlines Inc.,ANC,Ted Stevens Anchorage International Airport,Anchorage,SEA,Seattle-Tacoma International Airport,Seattle
1,AA,American Airlines Inc.,LAX,Los Angeles International Airport,Los Angeles,PBI,Palm Beach International Airport,West Palm Beach
2,US,US Airways Inc.,SFO,San Francisco International Airport,San Francisco,CLT,Charlotte Douglas International Airport,Charlotte
3,AA,American Airlines Inc.,LAX,Los Angeles International Airport,Los Angeles,MIA,Miami International Airport,Miami
4,AS,Alaska Airlines Inc.,SEA,Seattle-Tacoma International Airport,Seattle,ANC,Ted Stevens Anchorage International Airport,Anchorage
5,DL,Delta Air Lines Inc.,SFO,San Francisco International Airport,San Francisco,MSP,Minneapolis-Saint Paul International Airport,Minneapolis
6,NK,Spirit Air Lines,LAS,McCarran International Airport,Las Vegas,MSP,Minneapolis-Saint Paul International Airport,Minneapolis
7,US,US Airways Inc.,LAX,Los Angeles International Airport,Los Angeles,CLT,Charlotte Douglas International Airport,Charlotte
8,AA,American Airlines Inc.,SFO,San Francisco International Airport,San Francisco,DFW,Dallas/Fort Worth International Airport,Dallas-Fort Worth
9,DL,Delta Air Lines Inc.,LAS,McCarran International Airport,Las Vegas,ATL,Hartsfield-Jackson Atlanta International Airport,Atlanta


In [6]:
time_check = common.pd.read_sql("""
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'bronze'
  AND TABLE_NAME IN ('flights_new', 'weather')
  AND (
        COLUMN_NAME LIKE '%TIME%'
        OR COLUMN_NAME LIKE '%DATE%'
        OR COLUMN_NAME IN (
            'YEAR',
            'MONTH',
            'DAY',
            'ORIGIN_AIRPORT',
            'airport_code',
            'observation_time'
        )
      )
ORDER BY TABLE_NAME, ORDINAL_POSITION
""", engine)

time_check

,TABLE_NAME,COLUMN_NAME,DATA_TYPE
0,flights_new,MONTH,bigint
1,flights_new,DAY,bigint
2,flights_new,ORIGIN_AIRPORT,varchar
3,flights_new,DEPARTURE_TIME,float
4,flights_new,SCHEDULED_TIME,float
5,flights_new,ELAPSED_TIME,float
6,flights_new,AIR_TIME,float
7,flights_new,ARRIVAL_TIME,float
8,flights_new,SCHEDULED_DEPARTURE_DATETIME,datetime
9,weather,airport_code,varchar


In [7]:
time_sample = common.pd.read_sql("""
SELECT TOP 10
    f.ORIGIN_AIRPORT,
    f.SCHEDULED_DEPARTURE_DATETIME,
    w.observation_time
FROM bronze.flights_new f

LEFT JOIN bronze.weather w
    ON f.ORIGIN_AIRPORT = w.airport_code
    AND w.observation_time = DATEADD(
        HOUR,
        DATEDIFF(HOUR, 0, f.SCHEDULED_DEPARTURE_DATETIME),
        0
    )

WHERE f.SCHEDULED_DEPARTURE_DATETIME IS NOT NULL
ORDER BY f.SCHEDULED_DEPARTURE_DATETIME
""", engine)

time_sample

,ORIGIN_AIRPORT,SCHEDULED_DEPARTURE_DATETIME,observation_time
0,ANC,2015-01-01 00:05:00,2015-01-01
1,LAX,2015-01-01 00:10:00,2015-01-01
2,SFO,2015-01-01 00:20:00,2015-01-01
3,LAX,2015-01-01 00:20:00,2015-01-01
4,SEA,2015-01-01 00:25:00,2015-01-01
5,SFO,2015-01-01 00:25:00,2015-01-01
6,LAS,2015-01-01 00:25:00,2015-01-01
7,LAX,2015-01-01 00:30:00,2015-01-01
8,SFO,2015-01-01 00:30:00,2015-01-01
9,LAS,2015-01-01 00:30:00,2015-01-01


In [8]:
import pyarrow.parquet as pq

weather_file = pq.ParquetFile(
    "/workspaces/Flight-delay-analysis-E2E/data/api/weather_2015_all.parquet"
)

print(
    "observation_time tipi:",
    weather_file.schema_arrow.field("observation_time")
)

print(
    "Vaxtla əlaqəli sütunlar:",
    [
        col for col in weather_file.schema.names
        if "time" in col.lower()
        or "zone" in col.lower()
        or "utc" in col.lower()
    ]
)

observation_time tipi: pyarrow.Field<observation_time: timestamp[us]>
Vaxtla əlaqəli sütunlar: ['observation_time']


In [2]:
from timezonefinder import TimezoneFinder

tf = TimezoneFinder()

airports_tz = common.pd.read_csv(
    "/workspaces/Flight-delay-analysis-E2E/data/Flight/airports.csv"
)

airports_tz["TIMEZONE"] = [
    tf.timezone_at(lat=lat, lng=lon)
    if common.pd.notna(lat) and common.pd.notna(lon)
    else None
    for lat, lon in zip(
        airports_tz["LATITUDE"],
        airports_tz["LONGITUDE"]
    )
]

airports_tz[
    ["IATA_CODE", "CITY", "STATE", "TIMEZONE"]
].head(10)

,IATA_CODE,CITY,STATE,TIMEZONE
0,ABE,Allentown,PA,America/New_York
1,ABI,Abilene,TX,America/Chicago
2,ABQ,Albuquerque,NM,America/Denver
3,ABR,Aberdeen,SD,America/Chicago
4,ABY,Albany,GA,America/New_York
5,ACK,Nantucket,MA,America/New_York
6,ACT,Waco,TX,America/Chicago
7,ACV,Arcata/Eureka,CA,America/Los_Angeles
8,ACY,Atlantic City,NJ,America/New_York
9,ADK,Adak,AK,America/Adak


In [3]:
timezone_list = (
    airports_tz["TIMEZONE"]
    .value_counts(dropna=False)
)

timezone_list


TIMEZONE
America/Chicago                 102
America/New_York                 87
America/Denver                   36
America/Los_Angeles              35
America/Detroit                  14
America/Anchorage                 9
America/Boise                     5
Pacific/Honolulu                  5
America/Phoenix                   4
America/Sitka                     4
America/Puerto_Rico               3
NaN                               3
America/Indiana/Indianapolis      3
America/Juneau                    2
America/Nome                      2
America/St_Thomas                 2
America/Adak                      1
Etc/GMT+10                        1
America/Menominee                 1
Etc/GMT+11                        1
America/Kentucky/Louisville       1
America/Yakutat                   1
Name: count, dtype: int64

In [4]:
airports_tz.loc[
    airports_tz["TIMEZONE"].isna(),
    ["IATA_CODE", "CITY", "STATE", "LATITUDE", "LONGITUDE"]
]

,IATA_CODE,CITY,STATE,LATITUDE,LONGITUDE
96,ECP,Panama City,FL,NaN,NaN
234,PBG,Plattsburgh,NY,NaN,NaN
313,UST,St. Augustine,FL,NaN,NaN


In [5]:
missing_tz_usage = common.pd.read_sql("""
SELECT
    ORIGIN_AIRPORT,
    COUNT_BIG(*) AS flight_count
FROM bronze.flights_new
WHERE ORIGIN_AIRPORT IN ('ECP', 'PBG', 'UST')
GROUP BY ORIGIN_AIRPORT
ORDER BY flight_count DESC
""", engine)

missing_tz_usage

,ORIGIN_AIRPORT,flight_count
0,ECP,4521
1,PBG,281
2,UST,155


In [6]:
manual_timezones = {
    "ECP": "America/Chicago",
    "PBG": "America/New_York",
    "UST": "America/New_York"
}

airports_tz["TIMEZONE"] = airports_tz["TIMEZONE"].fillna(
    airports_tz["IATA_CODE"].map(manual_timezones)
)

print("Timezone tapılmayan:", airports_tz["TIMEZONE"].isna().sum())

Timezone tapılmayan: 0


In [7]:
tz_test = common.pd.read_sql("""
SELECT
    CAST('2015-01-01T10:00:00' AS datetime2)
        AT TIME ZONE 'Eastern Standard Time'
        AT TIME ZONE 'UTC' AS utc_time
""", engine)

tz_test

,utc_time
0,2015-01-01 15:00:00+00:00


In [8]:
iana_to_windows = {
    "America/Chicago": "Central Standard Time",
    "America/New_York": "Eastern Standard Time",
    "America/Detroit": "Eastern Standard Time",
    "America/Indiana/Indianapolis": "Eastern Standard Time",
    "America/Kentucky/Louisville": "Eastern Standard Time",

    "America/Denver": "Mountain Standard Time",
    "America/Boise": "Mountain Standard Time",
    "America/Phoenix": "US Mountain Standard Time",

    "America/Los_Angeles": "Pacific Standard Time",

    "America/Anchorage": "Alaskan Standard Time",
    "America/Sitka": "Alaskan Standard Time",
    "America/Juneau": "Alaskan Standard Time",
    "America/Nome": "Alaskan Standard Time",
    "America/Yakutat": "Alaskan Standard Time",

    "America/Adak": "Aleutian Standard Time",
    "Pacific/Honolulu": "Hawaiian Standard Time",
    "Etc/GMT+10": "Hawaiian Standard Time",
    "Etc/GMT+11": "UTC-11",

    "America/Puerto_Rico": "SA Western Standard Time",
    "America/St_Thomas": "SA Western Standard Time",

    "America/Menominee": "Central Standard Time"
}

airports_tz["WINDOWS_TZ"] = airports_tz["TIMEZONE"].map(iana_to_windows)

print(
    "SQL timezone tapılmayan:",
    airports_tz["WINDOWS_TZ"].isna().sum()
)

SQL timezone tapılmayan: 0


In [9]:
airport_timezones = airports_tz[
    ["IATA_CODE", "WINDOWS_TZ"]
].drop_duplicates()

airport_timezones.to_sql(
    "airport_timezones",
    engine,
    schema="bronze",
    if_exists="replace",
    index=False
)

print("SQL-ə yüklənən airport timezone sayı:", len(airport_timezones))

SQL-ə yüklənən airport timezone sayı: 322


In [10]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
    CREATE OR ALTER VIEW bronze.v_flights_utc AS

    SELECT
        f.*,
        tz.WINDOWS_TZ,

        CASE
            WHEN f.SCHEDULED_DEPARTURE_DATETIME IS NOT NULL
             AND tz.WINDOWS_TZ IS NOT NULL
            THEN CAST(
                f.SCHEDULED_DEPARTURE_DATETIME
                    AT TIME ZONE tz.WINDOWS_TZ
                    AT TIME ZONE 'UTC'
                AS datetime2
            )
        END AS SCHEDULED_DEPARTURE_UTC

    FROM bronze.flights_new f

    LEFT JOIN bronze.airport_timezones tz
        ON f.ORIGIN_AIRPORT = tz.IATA_CODE
    """))

print("Bütün flights üçün UTC view yaradıldı")

Bütün flights üçün UTC view yaradıldı


In [11]:
utc_check = common.pd.read_sql("""
SELECT
    COUNT_BIG(*) AS total_flights,
    SUM(
        CASE
            WHEN SCHEDULED_DEPARTURE_UTC IS NULL THEN 1
            ELSE 0
        END
    ) AS utc_missing
FROM bronze.v_flights_utc
""", engine)

utc_check

,total_flights,utc_missing
0,5812365,1011


In [12]:
weather_duplicates = common.pd.read_sql("""
SELECT COUNT_BIG(*) AS duplicate_groups
FROM (
    SELECT
        airport_code,
        observation_time
    FROM bronze.weather
    GROUP BY airport_code, observation_time
    HAVING COUNT(*) > 1
) x
""", engine)

weather_duplicates

,duplicate_groups
0,0


In [13]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
    CREATE OR ALTER VIEW bronze.v_flights_joined AS

    SELECT
        f.*,

        a.AIRLINE AS AIRLINE_NAME,

        o.AIRPORT AS ORIGIN_AIRPORT_NAME,
        o.CITY AS ORIGIN_CITY,
        o.STATE AS ORIGIN_STATE,
        o.COUNTRY AS ORIGIN_COUNTRY,

        d.AIRPORT AS DESTINATION_AIRPORT_NAME,
        d.CITY AS DESTINATION_CITY,
        d.STATE AS DESTINATION_STATE,
        d.COUNTRY AS DESTINATION_COUNTRY,

        w.observation_time AS WEATHER_OBSERVATION_UTC,
        w.temperature,
        w.precipitation,
        w.wind_speed,
        w.visibility,
        w.weather_condition,
        w.surface_pressure,
        w.wind_direction,
        w.snowfall,
        w.wind_gusts

    FROM bronze.v_flights_utc f

    LEFT JOIN bronze.airlines a
        ON f.AIRLINE = a.IATA_CODE

    LEFT JOIN bronze.airports o
        ON f.ORIGIN_AIRPORT = o.IATA_CODE

    LEFT JOIN bronze.airports d
        ON f.DESTINATION_AIRPORT = d.IATA_CODE

    LEFT JOIN bronze.weather w
        ON f.ORIGIN_AIRPORT = w.airport_code
        AND w.observation_time =
            DATEADD(
                HOUR,
                DATEDIFF(HOUR, 0, f.SCHEDULED_DEPARTURE_UTC),
                0
            )
    """))

print("Final JOIN view yaradıldı")

Final JOIN view yaradıldı


In [17]:
final_preview = common.pd.read_sql("""
SELECT TOP 20 *
FROM bronze.v_flights_joined
""", engine)

final_preview

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,...,WEATHER_OBSERVATION_UTC,temperature,precipitation,wind_speed,visibility,weather_condition,surface_pressure,wind_direction,snowfall,wind_gusts
0,1,1,4,AS,98,N407AS,ANC,SEA,5,2354.0,...,2015-01-01 09:00:00,-0.1,0.0,4.4,None,3.0,1017.6,81.0,0.0,9.7
1,1,1,4,HA,33,N385HA,LAX,OGG,1115,1117.0,...,2015-01-01 19:00:00,11.4,0.0,8.2,None,0.0,1015.6,119.0,0.0,19.8
2,1,1,4,MQ,3339,N524MQ,ORD,SYR,1335,1353.0,...,NaT,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
3,1,1,4,US,519,N178US,PDX,PHX,845,842.0,...,NaT,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
4,1,1,4,MQ,3402,N510MQ,LGA,CMH,1335,1328.0,...,2015-01-01 18:00:00,1.0,0.0,23.7,None,0.0,1017.7,244.0,0.0,46.1
5,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,2.0,...,2015-01-01 08:00:00,2.7,0.0,8.0,None,0.0,1012.2,8.0,0.0,15.1
6,1,1,4,AA,97,N3ENAA,DCA,DFW,845,842.0,...,NaT,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
7,1,1,4,MQ,3043,N644MQ,ORD,FNT,1115,1109.0,...,NaT,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
8,1,1,4,AA,317,N3MBAA,LGA,ORD,845,841.0,...,2015-01-01 13:00:00,-4.3,0.0,13.3,None,0.0,1021.6,251.0,0.0,32.8
9,1,1,4,MQ,3423,N612MQ,ORD,ALO,1335,1447.0,...,NaT,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN


In [18]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        IF OBJECT_ID('bronze.flights_joined', 'U') IS NOT NULL
            DROP TABLE bronze.flights_joined;

        SELECT *
        INTO bronze.flights_joined
        FROM bronze.v_flights_joined;
    """))

print("Yekun JOIN cədvəli Bronze-a yazıldı")

OperationalError: (pymssql.exceptions.OperationalError) (20017, b'DB-Lib error message 20017, severity 9:\nUnexpected EOF from the server\n')
[SQL: 
        IF OBJECT_ID('bronze.flights_joined', 'U') IS NOT NULL
            DROP TABLE bronze.flights_joined;

        SELECT *
        INTO bronze.flights_joined
        FROM bronze.v_flights_joined;
    ]
(Background on this error at: https://sqlalche.me/e/21/e3q8)

In [20]:
table_check = common.pd.read_sql("""
SELECT
    CASE
        WHEN OBJECT_ID('bronze.flights_joined', 'U') IS NOT NULL
        THEN 1
        ELSE 0
    END AS table_exists
""", engine)

table_check

,table_exists
0,0


In [21]:
from sqlalchemy import text

# Köhnə yarımçıq cədvəl varsa sil və boş yekun cədvəl yarat
with engine.begin() as conn:
    conn.execute(text("""
        IF OBJECT_ID('bronze.flights_joined', 'U') IS NOT NULL
            DROP TABLE bronze.flights_joined;

        SELECT TOP 0 *
        INTO bronze.flights_joined
        FROM bronze.v_flights_joined;
    """))

# Bütün ayları avtomatik yaz
for month in range(1, 13):
    with engine.begin() as conn:
        conn.execute(
            text("""
                INSERT INTO bronze.flights_joined
                SELECT *
                FROM bronze.v_flights_joined
                WHERE MONTH = :month
            """),
            {"month": month}
        )

    print(f"{month}-ci ay Bronze-a yazıldı")

print("Bütün yekun JOIN cədvəli Bronze-a yazıldı ✅")

1-ci ay Bronze-a yazıldı
2-ci ay Bronze-a yazıldı
3-ci ay Bronze-a yazıldı
4-ci ay Bronze-a yazıldı
5-ci ay Bronze-a yazıldı
6-ci ay Bronze-a yazıldı
7-ci ay Bronze-a yazıldı
8-ci ay Bronze-a yazıldı
9-ci ay Bronze-a yazıldı
10-ci ay Bronze-a yazıldı
11-ci ay Bronze-a yazıldı
12-ci ay Bronze-a yazıldı
Bütün yekun JOIN cədvəli Bronze-a yazıldı ✅


In [22]:
final_count = common.pd.read_sql("""
SELECT COUNT_BIG(*) AS row_count
FROM bronze.flights_joined
""", engine)

final_count

,row_count
0,5812365
